# Regressione logistica

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, make_scorer
import math
import tabulate
from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare treining
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training

In [ ]:
def training(file_path, csv_name):
    # Lettura dati
    df = pd.read_csv(file_path)

    # Definizione target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']

    # Filtro pazienti validi
    df_validi = df.dropna(subset=original_target_list).copy()

    # Binarizzazione target
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)

    final_target_list = ['PR_class', 'ER_class', 'KI67_class']

    # Preparazione feature
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    # Riempimento NaN
    features = features.fillna(features.mean())

    # Cross Validation (GroupKFold standard)
    cv = GroupKFold(n_splits=5)

    # Modello Base: Logistic Regression
    # Aumentiamo max_iter di base per evitare warning di mancata convergenza se non specificato
    base_model = LogisticRegression(random_state=42, n_jobs=1) 
    multi_output_model = MultiOutputClassifier(base_model)

    # Iperparametri
    iperparametri = {
        'estimator__C': [0.01],
        'estimator__penalty': ['l2'],
        'estimator__solver': ['saga'],
        'estimator__max_iter': [100],
        'estimator__fit_intercept': [True, False],
        'estimator__tol': [1e-4],
    }

    # Scorer (Zero Division Safe)
    def multi_f1_scorer(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        scores = []
        for i in range(y_true.shape[1]):
            scores.append(f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0))
        return np.mean(scores)

    scorer = make_scorer(multi_f1_scorer)

    # Calcolo combinazioni per info
    total_combinations = math.prod(len(v) for v in iperparametri.values())
    print(f"\nInizio Grid Search ({total_combinations} combinazioni) per: {csv_name}")

    # Configurazione Grid Search
    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=cv,
        scoring=scorer,
        n_jobs=-1,       # Parallelizzazione
        verbose=1,       # Output testuale
        refit=False,
        error_score='raise'
    )

    # Esecuzione
    grid_search.fit(features, target, groups=groups)

    # --- Riformattazione Output per compatibilità print_results ---
    scores = []
    results = grid_search.cv_results_
    
    for i in range(len(results['params'])):
        # 1. Pulisco chiavi (tolgo estimator__)
        params = {k.replace('estimator__', ''): v for k, v in results['params'][i].items()}
        
        # 2. Recupero fold scores per completezza (opzionale ma consigliato)
        current_fold_scores = [results[f'split{k}_test_score'][i] for k in range(5)]

        # 3. Creo dizionario risultato
        scores.append({
            **params,
            'mean_score': results['mean_test_score'][i],
            'std_score': results['std_test_score'][i],
            'fold_scores': current_fold_scores
        })

    return scores

# Stampo i risultati in un formato leggibile

In [ ]:
# Assumiamo che tu lo importi così, o come 'import tabulate' se usi tabulate.tabulate

def print_results(results_per_dataset):
    '''
    Stampa i risultati della Grid Search in modo organizzato usando tabulate
    SPECIFICO PER LOGISTIC REGRESSION
    '''
    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI")
    print("=" * 80)

    # Lista per il riepilogo finale comparativo
    summary_data = []

    for name, metrics_list in results_per_dataset.items():
        # Trova il risultato migliore in base al mean_score
        best_result = max(metrics_list, key=lambda x: x['mean_score'])

        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")

        # Tabella Iperparametri (Adattata per Logistic Regression)
        print("Iperparametri Ottimali:")
        params_table = [
            ['C', best_result.get('C')],
            ['penalty', best_result.get('penalty')],
            ['solver', best_result.get('solver')],
            ['max_iter', best_result.get('max_iter')],
            ['fit_intercept', best_result.get('fit_intercept')],
            ['tol', best_result.get('tol')]
        ]
        # Nota: uso tabulate.tabulate se importi "import tabulate", 
        # altrimenti solo tabulate se fai "from tabulate import tabulate"
        print(tabulate.tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))
        print()

        # Aggiungi al riepilogo comparativo
        summary_data.append([
            name,
            best_result['mean_score'], # Tengo float per ordinamento
            best_result['std_score'],
            best_result.get('C'),
            best_result.get('solver'),
            best_result.get('max_iter'),
            best_result.get('fit_intercept')
        ])

    # Riepilogo Comparativo Finale
    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")

    # Ordina per F1-score decrescente (elemento indice 1)
    summary_data.sort(key=lambda x: float(x[1]), reverse=True)

    print(tabulate.tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std Dev', 'C', 'Solver', 'Max Iter', 'Fit Intercept'],
                   tablefmt='grid',
                   floatfmt=('', '.3f', '.3f', '.4f', '', '', '')))


# Lettura dei file

In [ ]:
# Eseguo il training per tutti i dataset
results_per_dataset = {}

for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Stampo i risultati con tabulate
print_results(results_per_dataset)